In [1]:
import pandas as pd
import numpy as np

In [2]:
from tqdm import tqdm
import time

### get mapping file for picrust2 

In [4]:
genome_id = pd.read_csv("data/genome_id.txt", header=None)
genome_id = genome_id.iloc[:,0].values

In [5]:
BiGG_gene_predict = []
missing = []
for i in tqdm(range(len(genome_id))):
    try:
        temp = pd.read_csv(f"data/blast_output_bigg/{genome_id[i]}.tsv")
        temp.loc[:, "genome"] = [genome_id[i]] * temp.shape[0]
        BiGG_gene_predict.append(temp)
    except FileNotFoundError:
        missing.append(genome_id[i])
print(f"{len(missing)} genomes have no hit table")

100%|██████████| 26868/26868 [03:21<00:00, 133.67it/s]


In [6]:
BiGG_df = pd.concat(BiGG_gene_predict)

In [7]:
BiGG_gene = BiGG_df.BiGG_gene.unique()
genome = BiGG_df.genome.unique()

In [8]:
BiGG_df.head()

,query_gene,BiGG_gene,score,genome
0,CABKKDON_00644,STM_v1_0.STM0047,76.3,GCA_000007325.1
1,CABKKDON_00762,STM_v1_0.STM0054,362.0,GCA_000007325.1
2,CABKKDON_01907,STM_v1_0.STM0057,306.0,GCA_000007325.1
3,CABKKDON_01910,STM_v1_0.STM0059,82.0,GCA_000007325.1
4,CABKKDON_01912,STM_v1_0.STM0061,561.0,GCA_000007325.1


In [9]:
mapping_bigg_gene_table = pd.DataFrame(data = np.zeros((len(genome), len(BiGG_gene))),
                                       index = genome, columns = BiGG_gene)
mapping_scores_table = pd.DataFrame(data = np.zeros((len(genome), len(BiGG_gene))),
                                       index = genome, columns = BiGG_gene)

In [10]:
BiGG_df = BiGG_df.set_index("genome")

In [11]:
for i in tqdm(genome):
    diamond_table = BiGG_df.loc[i]
    mapping_bigg_gene_table.loc[i, diamond_table.BiGG_gene.values] = 1
    mapping_scores_table.loc[i, diamond_table.BiGG_gene.values] = diamond_table.score.values

100%|██████████| 26855/26855 [06:15<00:00, 71.45it/s]


In [16]:
mapping_bigg_gene_table.index.name = 'assembly'
mapping_scores_table.index.name = 'assembly'

In [13]:
mapping_scores_table.head()

,STM_v1_0.STM0047,STM_v1_0.STM0054,STM_v1_0.STM0057,STM_v1_0.STM0059,STM_v1_0.STM0061,STM_v1_0.STM0125,STM_v1_0.STM0129,STM_v1_0.STM0134,STM_v1_0.STM0202,STM_v1_0.STM0203,...,iMM904.YCL025C,iIS312.TCDM_13070,iAM_Pb448.PBANKA_010780,iMM1415.24060,iMM904.YEL058W,iMM904.YOR155C,iMM1415.69019,iAM_Pv461.PVX_119690,iAM_Pc455.PCYB_115080,iMM904.YIL002C
assembly,,,,,,,,,,,,,,,,,,,,,
GCA_000007325.1,76.3,362.0,306.0,82.0,561.0,293.0,282.0,179.0,417.0,176.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
GCA_000008885.1,88.6,0.0,0.0,0.0,0.0,508.0,417.0,327.0,443.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
GCA_000009845.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
GCA_000010565.1,70.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
GCA_000011445.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
mapping_bigg_gene_table.to_csv("data/mapping_bigg_gene_table.tsv",
                               sep="\t")

In [15]:
mapping_scores_table.to_csv("data/mapping_scores_table.tsv",
                               sep="\t")

### get OTUs metabolic model

In [18]:
embedding = pd.read_csv("../../data/social_niche_embedding_100.txt", 
                        header=None, sep=" ", low_memory=False, index_col=0)
embedding = embedding.drop("<unk>")
fid = embedding.index.values
# In the hsp.py outputs every U of an OTU ID was written as T (e.g. ABOU02000049 -> ABOT02000049);
# map those IDs back to the SNE IDs. The mapping is unambiguous for this embedding.
to_sne_id = {f.replace("U", "T"): f for f in fid}
assert len(to_sne_id) == len(fid)

In [68]:
bigg_gene_predicted_fid = pd.read_csv("data/bigg_gene_predicted.tsv",
                                      index_col=0, sep="\t", low_memory=False)
bigg_gene_predicted_fid.index = bigg_gene_predicted_fid.index.map(lambda x: to_sne_id.get(x, x))
bigg_gene_predicted_fid = bigg_gene_predicted_fid.loc[bigg_gene_predicted_fid.metadata_NSTI < 2]
inter_id = np.intersect1d(fid, bigg_gene_predicted_fid.index.values)
bigg_gene_predicted_fid = bigg_gene_predicted_fid.loc[inter_id]
bigg_gene_predicted_fid = bigg_gene_predicted_fid.drop(columns='metadata_NSTI')

In [69]:
scores_predicted_fid = pd.read_csv("data/scores_predicted.tsv",
                                      index_col=0, sep="\t", low_memory=False)
scores_predicted_fid.index = scores_predicted_fid.index.map(lambda x: to_sne_id.get(x, x))
scores_predicted_fid = scores_predicted_fid.loc[scores_predicted_fid.metadata_NSTI < 2]
inter_id = np.intersect1d(fid, scores_predicted_fid.index.values)
scores_predicted_fid = scores_predicted_fid.loc[inter_id]
scores_predicted_fid = scores_predicted_fid.drop(columns='metadata_NSTI')

In [88]:
bigg_gene = np.intersect1d(bigg_gene_predicted_fid.columns.values, scores_predicted_fid.columns.values)

In [71]:
scores_predicted_fid = scores_predicted_fid.loc[:, bigg_gene]
bigg_gene_predicted_fid = bigg_gene_predicted_fid.loc[:, bigg_gene]

In [72]:
scores_predicted_fid.shape

(14039, 25818)

In [73]:
### query_gene	BiGG_gene	score

In [90]:
for i in bigg_gene_predicted_fid.index.values:
    scores_predicted = scores_predicted_fid.loc[i].values
    bigg_gene_predicted = bigg_gene_predicted_fid.loc[i].values
    num_bigg_gene = np.sum(bigg_gene_predicted != 0)
    query_gene = [f"gene_{k}" for k in range(num_bigg_gene)]
    otu_bigg_gene = pd.DataFrame({"query_gene": query_gene, 
                                  "BiGG_gene": bigg_gene[bigg_gene_predicted != 0],
                                  "score": scores_predicted[bigg_gene_predicted != 0]})
    otu_bigg_gene = otu_bigg_gene.loc[otu_bigg_gene.BiGG_gene != "closest_reference_genome"].reset_index(drop=True)
    # present by parsimony but with a zero predicted score -> not kept
    otu_bigg_gene = otu_bigg_gene.loc[otu_bigg_gene.score > 0]
    otu_bigg_gene.to_csv(f"data/OTU_bigg_gene/{i}.tsv")  # keeps the row index, like the committed tables